# 🎬 CogVideoX-5B-I2V (Super Otimizado para Colab)

Este notebook gera vídeos a partir de uma imagem + prompt, usando o modelo **CogVideoX** com uso mínimo de RAM/VRAM para evitar travamentos no Colab (GPU T4).


In [ ]:
# ✅ Instalar dependências
!pip install diffusers==0.33.1 transformers accelerate einops gradio ffmpeg-python safetensors --quiet

In [ ]:
# 🔑 Login Hugging Face
from huggingface_hub import login
login()

In [ ]:
# 📦 Imports e Setup
import os
import torch
from PIL import Image
import gradio as gr
from diffusers import (
    CogVideoXImageToVideoPipeline,
    AutoencoderKLCogVideoX,
    CogVideoXTransformer3DModel,
)
from diffusers.utils import export_to_video, load_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs("outputs", exist_ok=True)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

In [ ]:
# 🚀 Carregar modelo com offload apenas (sem mover tudo para GPU de uma vez)
transformer = CogVideoXTransformer3DModel.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    subfolder="transformer",
    torch_dtype=torch.float16
)
text_encoder = AutoencoderKLCogVideoX.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    subfolder="vae",
    torch_dtype=torch.float16
)
pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    text_encoder=text_encoder,
    transformer=transformer,
    torch_dtype=torch.float16
)
# ⚠️ Não usar .to(device) direto
pipe.enable_sequential_cpu_offload()

In [ ]:
# 🎥 Geração de vídeo com 12 frames
def generate_video(image_path, prompt):
    image = load_image(image_path).convert("RGB").resize((720, 480))
    result = pipe(
        image=image,
        prompt=prompt,
        guidance_scale=5,
        num_inference_steps=20,
        num_frames=12
    )
    frames = result.frames[0]
    out = "outputs/cogvideo_light.mp4"
    export_to_video(frames, out, fps=8)
    return out

In [ ]:
# 🖼️ Interface Gradio
with gr.Blocks() as demo:
    gr.Markdown("## CogVideoX (versão leve para Colab)")
    with gr.Row():
        img = gr.Image(type="filepath", label="Imagem")
        prm = gr.Textbox(label="Prompt (ex: 'sunset by the sea')")
    btn = gr.Button("🎬 Gerar")
    vid = gr.Video(label="🎞️ Vídeo")
    btn.click(fn=generate_video, inputs=[img, prm], outputs=vid)
    demo.launch(share=True)